In [56]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset    

In [57]:
df = pd.read_csv("/Users/dhrutamacm2/Desktop/PyToarch/fmnist_small.csv")

In [58]:
df.sample(5)

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
3030,4,0,0,0,0,0,0,0,0,0,...,0,0,0,131,132,110,62,0,0,0
3542,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
436,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3380,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2134,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [59]:
x = df.drop(columns = ['label'])
y = df['label']

In [60]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 42)

In [61]:
# scaling the features to between 0 and 1
x_train = x_train/255.0
x_test = x_test/255.0

In [62]:
x_train_tensor = torch.tensor(x_train.to_numpy(),dtype = torch.float32)
x_test_tensor = torch.tensor(x_test.to_numpy(),dtype = torch.float32)
y_train_tensor = torch.tensor(y_train.to_numpy(), dtype = torch.long)
y_test_tensor = torch.tensor(y_test.to_numpy(), dtype = torch.long)

In [63]:
# create custom dataset class
class customdata(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels
    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [64]:
train_data = customdata(x_train_tensor,y_train_tensor)
test_data = customdata(x_test_tensor,y_test_tensor)

In [65]:
train_data[389]

(tensor([0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0039, 0.0000, 0.0000, 0.5059, 0.5176, 0.4627, 0.6275, 0.0000, 0.0000,
         0.0078, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0039, 0.0078,
         0.0000, 0.0000, 0.0000, 0.0000, 1.0000, 0.9686, 0.9451, 1.0000, 0.1373,
         0.0000, 0.0000, 0.0000, 0.0039, 0.0039, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.1608, 0.7725, 0.9255, 0.8863, 0.8902, 0.9176,
         0.8941, 0.2902, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0078, 0.0000,
         0.0000, 0.3059, 0.7725, 1.0000, 0.9686, 0.9608, 0.9137, 0.9137, 0.9020,
         0.8980, 0.9373, 0.9843, 1.0000, 0.8863, 0.4588, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0

In [66]:
train_loader = DataLoader(train_data, batch_size = 32, shuffle = True)
test_loader = DataLoader(test_data, batch_size = 32, shuffle = False)

In [77]:
len(train_loader)

150

In [67]:
# define the ANN architecture
class ANN(nn.Module):
    def __init__(self,num_features):
        super().__init__()
        self.network = nn.Sequential(
              nn.Linear(num_features,128),
              nn.ReLU(),
              nn.Linear(128,64),
              nn.ReLU(),
              nn.Linear(64,10)
        )
    def forward(self,features):
        out = self.network(features)
        return out
    

In [68]:
learing_rate = 0.01
epochs = 25

In [69]:
loss_function = nn.CrossEntropyLoss()

In [70]:
model = ANN(x_train_tensor.shape[1])
optimizer = torch.optim.Adam(model.parameters(),lr = learing_rate)

In [73]:
for epoch in range(epochs):
    total_epoch_loss = 0
    for batch,labels in train_loader:
        output = model(batch)
        loss = loss_function(output,labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_epoch_loss += loss.item()
    avg_loss = total_epoch_loss/len(train_loader)
    print(f"epochs : {epoch+1}, loss : {avg_loss}")

epochs : 1, loss : 0.334850793381532
epochs : 2, loss : 0.2401037719845772
epochs : 3, loss : 0.25790084458887574
epochs : 4, loss : 0.23905960988253355
epochs : 5, loss : 0.24736407063901425
epochs : 6, loss : 0.22365276670083403
epochs : 7, loss : 0.24180026099085808
epochs : 8, loss : 0.22978384016702572
epochs : 9, loss : 0.24704722213248412
epochs : 10, loss : 0.20121064268052577
epochs : 11, loss : 0.217290380174915
epochs : 12, loss : 0.23814393140375614
epochs : 13, loss : 0.19796088843916854
epochs : 14, loss : 0.2261449486017227
epochs : 15, loss : 0.21272302286699415
epochs : 16, loss : 0.20415273708601792
epochs : 17, loss : 0.16897568348795175
epochs : 18, loss : 0.20410009232660134
epochs : 19, loss : 0.2054913400237759
epochs : 20, loss : 0.18600158472855885
epochs : 21, loss : 0.19718905640145143
epochs : 22, loss : 0.16725329662983615
epochs : 23, loss : 0.1830672484015425
epochs : 24, loss : 0.19556371662765742
epochs : 25, loss : 0.1611436887085438


In [74]:
model.eval()

ANN(
  (network): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [75]:
# evaluation code
total = 0
correct = 0

with torch.no_grad():

  for batch_features, batch_labels in test_loader:

    outputs = model(batch_features)

    _, predicted = torch.max(outputs, 1)

    total = total + batch_labels.shape[0]

    correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)

0.8175


In [76]:
len(test_loader)

38